<a href="https://colab.research.google.com/github/korkutanapa/DCASE2025_TASK2/blob/main/AE_RESULTS_DEV_BEARING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# If needed in Colab:
!pip install openpyxl -q

In [2]:
!pip install -q tensorflow keras openpyxl scikit-learn pandas numpy matplotlib

In [8]:
!rm -rf /content/*

In [9]:
# ============================================================
# NORMAL-ONLY AUTOENCODER FOR DEV_BEARING
#
# Train file: 990+10 normal training samples
# Test file : 200 samples, normal + anomaly
#
# Train: normal samples only
# Score: higher = more anomalous
# Evaluation: ROC-AUC on test labels
# ============================================================

# ============================================================
# 0. INSTALLS FOR GOOGLE COLAB
# ============================================================

!pip install -q numpy pandas scipy scikit-learn matplotlib openpyxl tensorflow keras joblib


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import rankdata

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import NearestNeighbors

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


# ============================================================
# 3. HARDWARE CHECK
# ============================================================

print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

if len(gpus) > 0:
    print("GPU available:")
    for gpu in gpus:
        print(gpu)
else:
    print("No GPU found. CPU is also enough for this dataset.")


# ============================================================
# 4. FILE PATHS
# ============================================================

train_file = "/content/cubical_mel_tda_features_dev_bearing_train.xlsx"
test_file  = "/content/cubical_mel_tda_features_dev_bearing_test.xlsx"

# If running inside ChatGPT sandbox, use:
# train_file = "/mnt/data/cubical_mel_tda_features_dev_bearing_train.xlsx"
# test_file  = "/mnt/data/cubical_mel_tda_features_dev_bearing_test.xlsx"


# ============================================================
# 5. SETTINGS
# ============================================================

LABEL_COL = "label"

NORMAL_LABEL = "normal"
ANOMALY_LABEL = "anomaly"

EXCLUDE_COLS = [
    "label",
    "file_id",
    "file_path",
    "filename",
    "path",
    "machine"
]

# Normal train split
NORMAL_TRAIN_SIZE = 0.80
NORMAL_VAL_SIZE = 0.20

# AutoEncoder grid
LATENT_DIMS = [4, 8, 16, 32]
NOISE_STDS = [0.00, 0.03]
SPARSE_L1_VALUES = [0.0, 1e-5]

# Training
BATCH_SIZE = 32
EPOCHS = 300
PATIENCE = 30

SCALER_TYPE = "standard"   # "standard" or "robust"

# Threshold for binary decision.
# ROC-AUC does not depend on this threshold.
THRESHOLD_QUANTILE = 0.95

EPS = 1e-8


# ============================================================
# 6. HELPER FUNCTIONS
# ============================================================

def encode_labels(label_series):
    labels = label_series.astype(str).str.lower().str.strip()

    y = labels.map({
        NORMAL_LABEL: 0,
        ANOMALY_LABEL: 1,
        "0": 0,
        "1": 1
    })

    if y.isna().any():
        print("Unmapped labels:")
        print(label_series[y.isna()].unique())
        raise ValueError("Some labels could not be mapped.")

    return y.astype(int).values


def get_common_numeric_features(train_df, test_df):
    train_numeric = train_df.select_dtypes(include=[np.number]).columns.tolist()
    test_numeric = test_df.select_dtypes(include=[np.number]).columns.tolist()

    common_numeric = sorted(list(set(train_numeric).intersection(set(test_numeric))))

    feature_cols = [
        c for c in common_numeric
        if c not in EXCLUDE_COLS
    ]

    # Remove constant features based on train data
    valid_features = []

    for col in feature_cols:
        values = train_df[col].replace([np.inf, -np.inf], np.nan)
        if values.nunique(dropna=True) > 1:
            valid_features.append(col)

    return valid_features


def prepare_scaler():
    if SCALER_TYPE == "standard":
        return StandardScaler()
    elif SCALER_TYPE == "robust":
        return RobustScaler()
    else:
        raise ValueError("SCALER_TYPE must be 'standard' or 'robust'.")


def build_autoencoder(input_dim, latent_dim=8, noise_std=0.03, sparse_l1=0.0):
    """
    Normal-only sparse denoising AutoEncoder.
    """

    inp = layers.Input(shape=(input_dim,), name="input_features")

    x = inp

    if noise_std > 0:
        x = layers.GaussianNoise(noise_std)(x)

    # Encoder
    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.15)(x)

    latent = layers.Dense(
        latent_dim,
        activation="relu",
        activity_regularizer=regularizers.l1(sparse_l1),
        name="latent_space"
    )(x)

    # Decoder
    x = layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(latent)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)

    out = layers.Dense(
        input_dim,
        activation="linear",
        name="reconstructed_features"
    )(x)

    autoencoder = models.Model(inp, out, name="normal_only_autoencoder")
    encoder = models.Model(inp, latent, name="encoder")

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse"
    )

    return autoencoder, encoder


def reconstruction_errors(autoencoder, X):
    X_hat = autoencoder.predict(X, verbose=0)

    sq_err = (X - X_hat) ** 2
    abs_err = np.abs(X - X_hat)

    return sq_err, abs_err, X_hat


def feature_reliability_weights(val_sq_err):
    """
    Normal-only feature weights.
    Features reconstructed stably under normal conditions get higher weight.
    No anomaly labels are used.
    """

    med = np.median(val_sq_err, axis=0)
    mad = np.median(np.abs(val_sq_err - med), axis=0)
    robust_scale = 1.4826 * mad + EPS

    reliability = 1.0 / (med + robust_scale + EPS)

    # Clip extreme weights
    lo = np.percentile(reliability, 5)
    hi = np.percentile(reliability, 95)

    reliability = np.clip(reliability, lo, hi)

    # Normalize to mean 1
    weights = reliability / np.mean(reliability)

    return weights, med, robust_scale


def empirical_percentile_scores(reference_scores, scores):
    """
    Converts raw scores to normal-calibrated percentile scores.
    Uses only normal validation reference scores.

    Higher value = more abnormal compared with normal validation distribution.
    """

    ref_sorted = np.sort(reference_scores)

    percentile = np.searchsorted(
        ref_sorted,
        scores,
        side="right"
    ) / len(ref_sorted)

    return percentile


def compute_score_dict(
    sq_err,
    abs_err,
    weights,
    val_sq_median,
    val_sq_scale
):
    """
    Multiple normal-only anomaly scores.
    Higher = more anomalous.
    """

    weight_sum = np.sum(weights)

    mse = np.mean(sq_err, axis=1)
    mae = np.mean(abs_err, axis=1)

    weighted_mse = np.sum(sq_err * weights, axis=1) / weight_sum
    weighted_mae = np.sum(abs_err * weights, axis=1) / weight_sum

    # Normal-calibrated robust z-error
    z_err = (sq_err - val_sq_median) / val_sq_scale
    z_err_pos = np.maximum(z_err, 0)

    zmean = np.mean(z_err_pos, axis=1)
    weighted_zmean = np.sum(z_err_pos * weights, axis=1) / weight_sum

    # Top-k z-error: anomaly may affect only a subset of features
    n_features = sq_err.shape[1]
    top_k = max(5, int(0.10 * n_features))

    sorted_z = np.sort(z_err_pos, axis=1)
    topk_zmean = np.mean(sorted_z[:, -top_k:], axis=1)

    return {
        "mse": mse,
        "mae": mae,
        "weighted_mse": weighted_mse,
        "weighted_mae": weighted_mae,
        "zmean": zmean,
        "weighted_zmean": weighted_zmean,
        "topk_zmean": topk_zmean
    }


def latent_scores(encoder, X_ref_normal, X_eval, k=10):
    """
    Latent-space anomaly scores:
    1. Mahalanobis distance
    2. kNN distance in latent space

    Reference is normal training/validation latent data only.
    """

    Z_ref = encoder.predict(X_ref_normal, verbose=0)
    Z_eval = encoder.predict(X_eval, verbose=0)

    # Mahalanobis distance with shrinkage covariance
    cov = LedoitWolf().fit(Z_ref)
    mahal = cov.mahalanobis(Z_eval)

    # kNN distance in latent space
    k = min(k, len(Z_ref))

    knn = NearestNeighbors(
        n_neighbors=k,
        metric="euclidean"
    )
    knn.fit(Z_ref)

    dists, _ = knn.kneighbors(Z_eval)
    knn_dist = dists.mean(axis=1)

    return {
        "latent_mahalanobis": mahal,
        "latent_knn": knn_dist
    }


def evaluate_score(y_true, test_score, val_normal_score, threshold_quantile=0.95):
    """
    Uses validation-normal scores only for threshold.
    Uses test labels only for evaluation.
    """

    auc = roc_auc_score(y_true, test_score)

    threshold = np.quantile(val_normal_score, threshold_quantile)

    y_pred = (test_score >= threshold).astype(int)

    return {
        "ROC_AUC": auc,
        "effective_ROC_AUC_diagnostic": max(auc, 1 - auc),
        "threshold": threshold,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Confusion_Matrix": confusion_matrix(y_true, y_pred).tolist()
    }, y_pred


# ============================================================
# 7. LOAD DATA
# ============================================================

train_df = pd.read_excel(train_file)
test_df = pd.read_excel(test_file)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain label distribution:")
print(train_df[LABEL_COL].value_counts())

print("\nTest label distribution:")
print(test_df[LABEL_COL].value_counts())

feature_cols = get_common_numeric_features(train_df, test_df)

print("\nNumber of common usable numeric features:", len(feature_cols))


# ============================================================
# 8. PREPARE TRAIN AND TEST MATRICES
# ============================================================

y_train_file = encode_labels(train_df[LABEL_COL])
y_test = encode_labels(test_df[LABEL_COL])

# Train file should be all normal
print("\nTrain encoded labels:", np.bincount(y_train_file))
print("Test encoded labels :", np.bincount(y_test))

X_train_all_raw = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
X_test_raw = test_df[feature_cols].replace([np.inf, -np.inf], np.nan)

# Use only normal samples from train file
X_normal_all_raw = X_train_all_raw.iloc[y_train_file == 0].copy()

X_normal_train_raw, X_normal_val_raw = train_test_split(
    X_normal_all_raw,
    train_size=NORMAL_TRAIN_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

print("\nNormal AE train:", X_normal_train_raw.shape)
print("Normal AE val  :", X_normal_val_raw.shape)


# ============================================================
# 9. FIT IMPUTER AND SCALER ON NORMAL TRAIN ONLY
# ============================================================

imputer = SimpleImputer(strategy="median")

X_normal_train_imp = imputer.fit_transform(X_normal_train_raw)
X_normal_val_imp = imputer.transform(X_normal_val_raw)
X_test_imp = imputer.transform(X_test_raw)

scaler = prepare_scaler()

X_normal_train = scaler.fit_transform(X_normal_train_imp)
X_normal_val = scaler.transform(X_normal_val_imp)
X_test = scaler.transform(X_test_imp)

input_dim = X_normal_train.shape[1]

print("\nInput dimension:", input_dim)


# ============================================================
# 10. TRAIN AE GRID AND EVALUATE
# ============================================================

all_results = []
all_score_tables = []
all_histories = []

for latent_dim in LATENT_DIMS:

    for noise_std in NOISE_STDS:

        for sparse_l1 in SPARSE_L1_VALUES:

            print("\n" + "=" * 100)
            print(
                f"Training AE | latent_dim={latent_dim}, "
                f"noise_std={noise_std}, sparse_l1={sparse_l1}"
            )
            print("=" * 100)

            tf.keras.backend.clear_session()

            autoencoder, encoder = build_autoencoder(
                input_dim=input_dim,
                latent_dim=latent_dim,
                noise_std=noise_std,
                sparse_l1=sparse_l1
            )

            early_stop = callbacks.EarlyStopping(
                monitor="val_loss",
                mode="min",
                patience=PATIENCE,
                restore_best_weights=True
            )

            reduce_lr = callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                mode="min",
                factor=0.5,
                patience=10,
                min_lr=1e-5
            )

            history = autoencoder.fit(
                X_normal_train,
                X_normal_train,
                validation_data=(X_normal_val, X_normal_val),
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                callbacks=[early_stop, reduce_lr],
                verbose=0
            )

            print("Best val loss:", np.min(history.history["val_loss"]))
            print("Epochs trained:", len(history.history["loss"]))

            # ------------------------------------------------
            # Reconstruction errors
            # ------------------------------------------------

            val_sq_err, val_abs_err, _ = reconstruction_errors(
                autoencoder,
                X_normal_val
            )

            test_sq_err, test_abs_err, _ = reconstruction_errors(
                autoencoder,
                X_test
            )

            weights, val_sq_median, val_sq_scale = feature_reliability_weights(
                val_sq_err
            )

            val_score_dict = compute_score_dict(
                sq_err=val_sq_err,
                abs_err=val_abs_err,
                weights=weights,
                val_sq_median=val_sq_median,
                val_sq_scale=val_sq_scale
            )

            test_score_dict = compute_score_dict(
                sq_err=test_sq_err,
                abs_err=test_abs_err,
                weights=weights,
                val_sq_median=val_sq_median,
                val_sq_scale=val_sq_scale
            )

            # ------------------------------------------------
            # Latent scores
            # ------------------------------------------------

            latent_val_scores = latent_scores(
                encoder=encoder,
                X_ref_normal=X_normal_train,
                X_eval=X_normal_val,
                k=10
            )

            latent_test_scores = latent_scores(
                encoder=encoder,
                X_ref_normal=X_normal_train,
                X_eval=X_test,
                k=10
            )

            val_score_dict.update(latent_val_scores)
            test_score_dict.update(latent_test_scores)

            # ------------------------------------------------
            # Normal-calibrated percentile scores
            # ------------------------------------------------

            val_percentile_scores = {}
            test_percentile_scores = {}

            for score_name in test_score_dict.keys():

                val_raw = val_score_dict[score_name]
                test_raw = test_score_dict[score_name]

                val_pct = empirical_percentile_scores(
                    reference_scores=val_raw,
                    scores=val_raw
                )

                test_pct = empirical_percentile_scores(
                    reference_scores=val_raw,
                    scores=test_raw
                )

                val_percentile_scores[score_name + "_pct"] = val_pct
                test_percentile_scores[score_name + "_pct"] = test_pct

            # Ensemble score:
            # average of normal-calibrated percentile scores
            val_ensemble = np.mean(
                np.vstack(list(val_percentile_scores.values())),
                axis=0
            )

            test_ensemble = np.mean(
                np.vstack(list(test_percentile_scores.values())),
                axis=0
            )

            val_score_dict["ensemble_pct"] = val_ensemble
            test_score_dict["ensemble_pct"] = test_ensemble

            # Also add percentile score versions
            val_score_dict.update(val_percentile_scores)
            test_score_dict.update(test_percentile_scores)

            # ------------------------------------------------
            # Evaluate each score
            # ------------------------------------------------

            for score_name in test_score_dict.keys():

                result, y_pred = evaluate_score(
                    y_true=y_test,
                    test_score=test_score_dict[score_name],
                    val_normal_score=val_score_dict[score_name],
                    threshold_quantile=THRESHOLD_QUANTILE
                )

                result_row = {
                    "latent_dim": latent_dim,
                    "noise_std": noise_std,
                    "sparse_l1": sparse_l1,
                    "score_name": score_name,
                    "ROC_AUC": result["ROC_AUC"],
                    "effective_ROC_AUC_diagnostic": result["effective_ROC_AUC_diagnostic"],
                    "Accuracy": result["Accuracy"],
                    "Precision": result["Precision"],
                    "Recall": result["Recall"],
                    "F1": result["F1"],
                    "threshold": result["threshold"],
                    "confusion_matrix": result["Confusion_Matrix"],
                    "best_val_loss": np.min(history.history["val_loss"]),
                    "n_epochs": len(history.history["loss"])
                }

                all_results.append(result_row)

                temp_scores = pd.DataFrame({
                    "latent_dim": latent_dim,
                    "noise_std": noise_std,
                    "sparse_l1": sparse_l1,
                    "score_name": score_name,
                    "y_true": y_test,
                    "score": test_score_dict[score_name],
                    "y_pred": y_pred
                })

                all_score_tables.append(temp_scores)

            # Training history
            hist_df = pd.DataFrame(history.history)
            hist_df["latent_dim"] = latent_dim
            hist_df["noise_std"] = noise_std
            hist_df["sparse_l1"] = sparse_l1
            hist_df["epoch"] = np.arange(1, len(hist_df) + 1)

            all_histories.append(hist_df)


# ============================================================
# 11. RESULTS SUMMARY
# ============================================================

results_df = pd.DataFrame(all_results)
scores_df = pd.concat(all_score_tables, ignore_index=True)
histories_df = pd.concat(all_histories, ignore_index=True)

print("\n" + "=" * 100)
print("TOP 30 RESULTS BY ROC-AUC")
print("=" * 100)

top_results = results_df.sort_values(
    "ROC_AUC",
    ascending=False
).reset_index(drop=True)

print(
    top_results[
        [
            "latent_dim",
            "noise_std",
            "sparse_l1",
            "score_name",
            "ROC_AUC",
            "effective_ROC_AUC_diagnostic",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "best_val_loss",
            "n_epochs"
        ]
    ].head(30)
)


print("\n" + "=" * 100)
print("BEST RESULT")
print("=" * 100)

best_result = top_results.iloc[0]

print(best_result)


print("\n" + "=" * 100)
print("MEAN ROC-AUC BY SCORE TYPE")
print("=" * 100)

score_summary = (
    results_df
    .groupby("score_name")
    .agg(
        mean_ROC_AUC=("ROC_AUC", "mean"),
        max_ROC_AUC=("ROC_AUC", "max"),
        median_ROC_AUC=("ROC_AUC", "median"),
        mean_F1=("F1", "mean")
    )
    .reset_index()
    .sort_values("mean_ROC_AUC", ascending=False)
)

print(score_summary)


# ============================================================
# 12. SAVE RESULTS
# ============================================================

results_df.to_csv("dev_bearing_autoencoder_all_results.csv", index=False)
top_results.to_csv("dev_bearing_autoencoder_top_results.csv", index=False)
scores_df.to_csv("dev_bearing_autoencoder_test_scores.csv", index=False)
score_summary.to_csv("dev_bearing_autoencoder_score_summary.csv", index=False)
histories_df.to_csv("dev_bearing_autoencoder_training_history.csv", index=False)

with pd.ExcelWriter("dev_bearing_autoencoder_results_summary.xlsx", engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="all_results", index=False)
    top_results.head(50).to_excel(writer, sheet_name="top50_results", index=False)
    score_summary.to_excel(writer, sheet_name="score_summary", index=False)
    histories_df.to_excel(writer, sheet_name="training_history", index=False)

print("\nSaved files:")
print("- dev_bearing_autoencoder_all_results.csv")
print("- dev_bearing_autoencoder_top_results.csv")
print("- dev_bearing_autoencoder_test_scores.csv")
print("- dev_bearing_autoencoder_score_summary.csv")
print("- dev_bearing_autoencoder_training_history.csv")
print("- dev_bearing_autoencoder_results_summary.xlsx")

TensorFlow: 2.20.0
GPU available:
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
Train shape: (1000, 276)
Test shape : (200, 276)

Train label distribution:
label
normal    1000
Name: count, dtype: int64

Test label distribution:
label
anomaly    100
normal     100
Name: count, dtype: int64

Number of common usable numeric features: 264

Train encoded labels: [1000]
Test encoded labels : [100 100]

Normal AE train: (800, 264)
Normal AE val  : (200, 264)

Input dimension: 264

Training AE | latent_dim=4, noise_std=0.0, sparse_l1=0.0
Best val loss: 0.24723726511001587
Epochs trained: 228

Training AE | latent_dim=4, noise_std=0.0, sparse_l1=1e-05
Best val loss: 0.26087486743927
Epochs trained: 188

Training AE | latent_dim=4, noise_std=0.03, sparse_l1=0.0
Best val loss: 0.24284213781356812
Epochs trained: 203

Training AE | latent_dim=4, noise_std=0.03, sparse_l1=1e-05
Best val loss: 0.24971500039100647
Epochs trained: 196

Training AE | latent_dim=8, noise_std=0.0, spa